# Распределение товаров по торговым точкам через роевой интеллект

## Библиотеки

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from time import time

## Эвристики

Влияние продаж

In [2]:
sales_factor = 0.001

Влияние загруженности

In [3]:
load_factor = 0.0009

Так как товаров очень много, то при умножении суммы на константу могут получится слишком большие значения, из-за чего обновление феромонов будет гигантским. Потому нужно брать влияния продаж и загруженности в пределах 0.001

Сколько итераций считать

In [4]:
iterations_count = 100

Ранняя остановка. Если лучшее решение не менялось на протяжении указанного кол-ва итераций, то останваливаем алгоритм. Значение 0 значит без ранней остановки

In [5]:
early_stopping = 10

Если нынешнее лучшее решение не больше предыдущего хотя бы на эпсилон, то лучшее решение не обновляется

In [6]:
epsilon = 10**-4

Максимальная скорость познавательного самообучения

In [7]:
max_speed_research = 2.05

Максимальная скорость социального самообучения

In [8]:
max_speed_social = 2.05

Коэффициент предела скорости. То есть новый предел скорости будет velocity_limit_mult * v, где v - объём товара в закрывающемся магазине

In [9]:
velocity_limit_mult = 0.2

Коэффициент начальной скорости. Не должен быть больше speed_limit_mult

In [10]:
initial_velocity_mult = 0.1

Коэффициент скорости выталкивания перегрузки. Скорость частицы уменьшается на эту скорость

In [11]:
overflow_velocity_mult = 0.1

Коэффициент перегрузки. Позиция частицы насильно сдвигается на значение перегрузку

In [12]:
overflow_position_mult = 0.2

Важно, чтобы сумма максимальный скоростей самообучения была больше 4

Коэффициент $\alpha ∈ (0, 1)$, контролирует близость коэффициента сужения к максимуму

In [13]:
alpha = 0.9
shrink = 2 * alpha / (max_speed_research + max_speed_social - 2)

Число частиц

In [14]:
particle_count = 100

## Подгрузка датасета

In [15]:
col_vol = "Объём товара в закрывающемся магазине (V)"
col_maxcap = "Максимальная вместимость товара в магазине (M)"
col_item = "Код товара"
col_store = "Магазин"
col_remain = "Остаток товара в магазине (R)"
col_sales = "Продажи товара в магазине (S)"
col_dist = "Распределение частиц"

In [16]:
items = pd.read_csv(r"./filtered_datasets/items_closing_store.csv", sep=";")
items = items.set_index([col_item, col_store])
items

Объём товара в закрывающемся магазине (V)  \
Код товара Магазин                                              
226249     54                                            10.0   
           56                                            10.0   
           57                                            10.0   
           60                                            10.0   
           61                                            10.0   
...                                                       ...   
239388     95                                            26.0   
239397     38                                            12.0   
239398     38                                            13.0   
239515     38                                            42.0   
239516     38                                            42.0   

                    Продажи товара в магазине (S)  \
Код товара Магазин                                  
226249     54                            1.379157   
           56                            0.943567   
           57                            0.857143   
           60                            0.406742   
           61                            0.713024   
...                                           ...   
239388     95                            1.500000   
239397     38                            1.116279   
239398     38                            2.023256   
239515     38                            1.796296   
239516     38                            1.740741   

                    Максимальная вместимость товара в магазине (M)  \
Код товара Магазин                                                   
226249     54                                                    8   
           56                                                    8   
           57                                                    8   
           60                                                    8   
           61                                                    8   
...                                                            ...   
239388     95                                                   25   
239397     38                                                   24   
239398     38                                                   24   
239515     38                                                   30   
239516     38                                                   24   

                    Остаток товара в магазине (R)  
Код товара Магазин                                 
226249     54                                16.0  
           56                                22.0  
           57                                19.0  
           60                                28.0  
           61                                22.0  
...                                           ...  
239388     95                                20.0  
239397     38                                21.0  
239398     38                                45.0  
239515     38                                35.0  
239516     38                                28.0  

[413689 rows x 4 columns]

Со всем датасетом работать очень медленно. Гораздо лучше будет сделать несколько матриц

$i$ - товар, $i ∈ [1, N]$, $N$ - кол-во товаров

$j$ - магазин, $j ∈ [1, M]$, $M$ - кол-во магазинов

*closing_volume* - вектор размерности $N×1$, показывает кол-во товара $i$ в закрывающемся магазине

*sales* - матрица размерности $N×M$, показывает продажи товара $i$ в магазине $j$

*maxcap* - матрица размерности $N×M$, показывает вместимость на полках товара $i$ в магазине $j$

*remains* - матрица размерности $N×M$, показывает остатки товара $i$ в магазине $j$

*pheromones* - матрица размерности $N×M$, показывает феромоны для распределения товара $i$ в магазин $j$

Если товар $i$ не числится в магазине $j$, то тогда соответственные значения $(i, j)$ во всех матрицах равны 0

In [17]:
items_id = items.index.get_level_values(0).unique().values
N = items_id.shape[0]
N

5366

In [18]:
stores_id = items.index.get_level_values(1).unique().values
M = stores_id.shape[0]
M

98

Переиндексация для создания пар $(i, j)$

In [19]:
full_index = pd.MultiIndex.from_product([items_id, stores_id], names=[col_item, col_store])
df_full = items.reindex(full_index)
df_full

Объём товара в закрывающемся магазине (V)  \
Код товара Магазин                                              
226249     54                                            10.0   
           56                                            10.0   
           57                                            10.0   
           60                                            10.0   
           61                                            10.0   
...                                                       ...   
239221     88                                             NaN   
           105                                            NaN   
           29                                             NaN   
           27                                             NaN   
           113                                           17.0   

                    Продажи товара в магазине (S)  \
Код товара Магазин                                  
226249     54                            1.379157   
           56                            0.943567   
           57                            0.857143   
           60                            0.406742   
           61                            0.713024   
...                                           ...   
239221     88                                 NaN   
           105                                NaN   
           29                                 NaN   
           27                                 NaN   
           113                           0.302326   

                    Максимальная вместимость товара в магазине (M)  \
Код товара Магазин                                                   
226249     54                                                  8.0   
           56                                                  8.0   
           57                                                  8.0   
           60                                                  8.0   
           61                                                  8.0   
...                                                            ...   
239221     88                                                  NaN   
           105                                                 NaN   
           29                                                  NaN   
           27                                                  NaN   
           113                                                10.0   

                    Остаток товара в магазине (R)  
Код товара Магазин                                 
226249     54                                16.0  
           56                                22.0  
           57                                19.0  
           60                                28.0  
           61                                22.0  
...                                           ...  
239221     88                                 NaN  
           105                                NaN  
           29                                 NaN  
           27                                 NaN  
           113                                5.0  

[525868 rows x 4 columns]

Превращаем всё теперь в матрицы

In [20]:
closing_volume = items.groupby(level=0, sort=False).first()[col_vol].values
sales = df_full[col_sales].values.reshape(N, M)
maxcap = df_full[col_maxcap].values.reshape(N, M)
remains = df_full[col_remain].values.reshape(N, M)

Заполняем пропуски нулями

In [21]:
sales = np.nan_to_num(sales, nan=0)
maxcap = np.nan_to_num(maxcap, nan=0)
remains = np.nan_to_num(remains, nan=0)

## Рой частиц

### Частица

Так как рой частиц работает с действительными числами, а распределение - это только целые числа, то нужно сначала округлить значения, а потом восстановить (т.е. repair)

repair работает следующим образом:

1. Сначала исходное распределение с действительными значениями округляем в меньшую сторону
2. Эта операция может лишить нас товаров, потому находим разницу между тем, сколько товаров распределить можно всего, и сколько распределили мы после округления
3. Затем считаем потерянную дробную часть и сортируем в порядке убывания. Таким образом самые большие потерянные дробные части идут впереди
4. Находим индексы тех распределений, где больше всего было потеряно товаров. В numpy шаг 3 и 4 делаются сразу через argsort
5. Теперь осталось добавить по предмету там, где нужнее всего
6. Если же разница товаров отрицательная, то в шаге 3 сортируем в порядке возрастания, чтобы получить самые большие добавленные части. Потом же просто отнимаем эти товары

In [22]:
def particle_function(particle_position):    
    x, v, s, m, r = particle_position, closing_volume, sales, maxcap, remains
    c = np.zeros((N, M), dtype=np.int64)  # само распределение

    for i in range(N):  # repair      
        c[i] = np.floor(x[i]).astype(np.int64)  # округление до нижней границы
        item_loss = int(v[i] - np.sum(c[i]))  # сколько товаров таким образом потеряли

        if item_loss > 0:
            fractional = x[i] - c[i]  # дробные части
            indexes = np.argsort(-fractional)  # сортировка по значениям дробных частей в порядке убывания
            c[i][indexes[:item_loss]] += 1  # докинули товары там, где это важнее всего
                
        elif item_loss < 0:
            fractional = x[i] - c[i]  # дробные части
            indexes = np.argsort(fractional)  # сортировка по значениям дробных частей в порядке возрастания
            c[i][indexes[:-item_loss]] -= 1  # убираем товары оттуда, где это важноо всего

    sales_rating = sales_factor * (c + r) * s / (1 + r)

    load_rating = np.clip((c + r - m), a_min=0.0, a_max=None)  # убираем штраф, если нет перегрузки
    np.divide(load_rating, m, out=load_rating, where=m > 0)  # чтобы не поделить на 0
    load_rating *= load_factor
    
    total_rating = np.sum(sales_rating - load_rating)  # оценка решения
    
    return total_rating, c

### Рой

In [23]:
def particle_swarm_optimization():
    # Здесь индексы None делают здесь из одномерного массива трёхмерные особым образом, разберём на примере
    # Было a = [1, 2, 3]. Сделали b = a[None, :, None]. Получили b = [[[1], [2], [3]]]. Теперь a[0] == b[0][0][0], a[1] == b[0][0][1] и т.д.
    
    particle_positions = np.random.rand(particle_count, N, M)  # случайные значения от 0 до 1 для начальных позиций
    sum_positions = np.sum(particle_positions, axis=2, keepdims=True)  # сумма по товарам
    particle_positions = np.divide(particle_positions, sum_positions, out=np.zeros((particle_count, N, M), dtype=np.float64), where= sum_positions != 0.0)  # потом делаем равномерное распределение, чтобы не превысить кол-во товара в закрывающемся магазине 
    particle_positions *= closing_volume[None, :, None]  # получим таким образом начальные позиции в границах товаров
    
    in_stock = (maxcap > 0)[None, :]  # т.е. здесь товары действительно находятся в продаже
    particle_positions *= in_stock  # чтобы не считать там, где товары не продаются

    max_velocity = closing_volume[None, :, None]*velocity_limit_mult  # предел скорости
    initial_velocity = closing_volume[None, :, None]*initial_velocity_mult  # предел начальной скорости
    particle_velocities = np.random.uniform(-initial_velocity, initial_velocity, size=(particle_count, N, M))  # начальная скорость
    particle_velocities *= in_stock  # чтобы не считать там, где товары не продаются
    
    best_positions = np.zeros((particle_count, N, M), dtype=np.float64)
    best_ratings = np.full(particle_count, np.nan)  # лучшие оценки лучших позиций
    best_distributions = np.zeros((particle_count, N, M), dtype=np.int64)  # лучшие распределения (целочисленные)
    
    neighbors = np.array([((i-1) % particle_count, (i+1) % particle_count) for i in range(particle_count)], dtype=np.int64)  # кольцевая топология - два соседа для каждой частицы
    
    no_improvements = 0
    total_best_rating, total_best_index = None, None

    for i in range(iterations_count):
        print(f"\nIteration {i+1} out of {iterations_count}")

        iter_time = time()

        for particle in range(particle_count):  # каждая частица делает repair, затем считает оценку своего решения
            particle_rating, particle_distribution = particle_function(particle_positions[particle])

            if np.isnan(best_ratings[particle]) or particle_rating > best_ratings[particle]:  # обновляем лучшие положения частиц
                best_ratings[particle] = particle_rating
                best_positions[particle] = particle_positions[particle]  # изначально, лучшее положение каждой частицы - это её начальное положение
                best_distributions[particle] = particle_distribution

        print(f"Ratings for {particle_count} particles updated, took: {time() - iter_time}")

        current_best_index = np.argmax(best_ratings)
        current_best_rating = best_ratings[current_best_index]
        no_improvements += 1

        print("Best rating for the current iteration:", current_best_rating)

        if total_best_rating is None or total_best_index is None or (current_best_rating - total_best_rating) >= epsilon:
            total_best_rating = current_best_rating
            total_best_index = current_best_index
            no_improvements = 0

        print("Total best rating:", total_best_rating)

        if early_stopping > 0 and no_improvements >= early_stopping:
            print("Early stopping")
            break

        iter_time = time()
        
        for particle in range(particle_count):  # считаем обновление скоростей и позиций
            left_neighbor, right_neighbor = neighbors[particle]
            best_neighbor = 0

            if best_ratings[left_neighbor] > best_ratings[right_neighbor]:  # находим индекс лучшего соседа
                best_neighbor = left_neighbor

            else:
                best_neighbor = right_neighbor

            b = best_positions[particle]
            h = best_positions[best_neighbor]
            f1 = np.random.rand(N, M) * max_speed_research
            f2 = np.random.rand(N, M) * max_speed_social

            # np.clip((particle_positions[particle] + remains - maxcap), a_min=0.0, a_max=None) - это и есть overflow, т. е. перегрузка

            particle_velocities[particle] = shrink * particle_velocities[particle] + f1 * (b - particle_positions[particle]) + f2 * (h - particle_positions[particle])  # формула изменения скорости
            particle_velocities[particle] -= np.clip((particle_positions[particle] + remains - maxcap), a_min=0.0, a_max=None) * overflow_velocity_mult  # скорость выталкивания частицы при перегрузке
            particle_velocities[particle] = np.clip(particle_velocities[particle], min=-max_velocity, max=max_velocity)  # ограничиваем скорость
            
            particle_positions[particle] += particle_velocities[particle]  # перемещение частицы
            particle_positions[particle] -= np.clip((particle_positions[particle] + remains - maxcap), a_min=0.0, a_max=None) * overflow_position_mult  # напрямую выталкиваем частицу при перегрузке
            particle_positions[particle] = np.clip(particle_positions[particle], a_min=0.0, a_max=None)  # позиции не должны быть отрицательными
            
            items_sum = np.sum(particle_positions[particle], axis=1, keepdims=True)  # сумма товаров по магазинам
            adjust_positions = np.divide(closing_volume[:, None], items_sum, out=np.zeros((N, M), dtype=np.float64), where= items_sum != 0.0)  # коэффициент, во сколько раз количество товара в закрывающемся магазине больше / меньше, чем в распределении
            particle_positions[particle] *= adjust_positions  # сдвигаем позиции так, чтобы их сумма была равна кол-ву товара в закрывающемся магазине
  
    return total_best_rating, best_distributions[total_best_index]    

In [24]:
time_start = time()
best_rating, best_solution = particle_swarm_optimization()


Iteration 1 out of 100
Ratings for 100 particles updated, took: 8.214381694793701
Best rating for the current iteration: 47.4075090574304
Total best rating: 47.4075090574304

Iteration 2 out of 100
Ratings for 100 particles updated, took: 7.790458679199219
Best rating for the current iteration: 56.824498278409735
Total best rating: 56.824498278409735

Iteration 3 out of 100
Ratings for 100 particles updated, took: 7.82852578163147
Best rating for the current iteration: 58.29985824278686
Total best rating: 58.29985824278686

Iteration 4 out of 100
Ratings for 100 particles updated, took: 7.770237922668457
Best rating for the current iteration: 59.23964260999487
Total best rating: 59.23964260999487

Iteration 5 out of 100
Ratings for 100 particles updated, took: 7.901870489120483
Best rating for the current iteration: 61.25129257464659
Total best rating: 61.25129257464659

Iteration 6 out of 100
Ratings for 100 particles updated, took: 7.835295915603638
Best rating for the current itera

In [25]:
print("Best rating:", best_rating)
print("Time taken:", time() - time_start)

Best rating: 73.14638382167688
Time taken: 922.0956757068634


## Выгрузка решения

In [26]:
def output_csv(df, filename, with_index=False):
    df_dirpath = Path(r"./solutions")
    df_dirpath.mkdir(parents=True, exist_ok=True)

    df_filepath = df_dirpath / filename

    with df_filepath.open("w") as f:
        df.to_csv(f, sep=";", lineterminator="\n", index=with_index)

In [27]:
df_solution = pd.DataFrame(best_solution, index=items_id, columns=stores_id).stack()
df_full[col_dist] = df_solution.reindex(df_full.index).values
df_full

Объём товара в закрывающемся магазине (V)  \
Код товара Магазин                                              
226249     54                                            10.0   
           56                                            10.0   
           57                                            10.0   
           60                                            10.0   
           61                                            10.0   
...                                                       ...   
239221     88                                             NaN   
           105                                            NaN   
           29                                             NaN   
           27                                             NaN   
           113                                           17.0   

                    Продажи товара в магазине (S)  \
Код товара Магазин                                  
226249     54                            1.379157   
           56                            0.943567   
           57                            0.857143   
           60                            0.406742   
           61                            0.713024   
...                                           ...   
239221     88                                 NaN   
           105                                NaN   
           29                                 NaN   
           27                                 NaN   
           113                           0.302326   

                    Максимальная вместимость товара в магазине (M)  \
Код товара Магазин                                                   
226249     54                                                  8.0   
           56                                                  8.0   
           57                                                  8.0   
           60                                                  8.0   
           61                                                  8.0   
...                                                            ...   
239221     88                                                  NaN   
           105                                                 NaN   
           29                                                  NaN   
           27                                                  NaN   
           113                                                10.0   

                    Остаток товара в магазине (R)  Распределение частиц  
Код товара Магазин                                                       
226249     54                                16.0                     0  
           56                                22.0                     0  
           57                                19.0                     0  
           60                                28.0                     0  
           61                                22.0                     0  
...                                           ...                   ...  
239221     88                                 NaN                     0  
           105                                NaN                     0  
           29                                 NaN                     0  
           27                                 NaN                     0  
           113                                5.0                     0  

[525868 rows x 5 columns]

In [28]:
output_csv(df_full.dropna(), "swarm_solution.csv", with_index=True)